# ABT (Analytical Base Table)

## Objetivo
Construir una tabla con una fila por cliente que combine variables del pasado (ventana de entrenamiento, ago 2018 – ene 2019) con el target del futuro (mar – abr 2019). El trabajo llega hasta la ABT completa, con la mayor cantidad de variables posibles, antes de la reducción de dimensionalidad.

## Entrada
- 02_data_train_limpio.parquet: 6 meses de entrenamiento, ya limpios (139.146 filas).
- 02_data_tarj.parquet: tipo de tarjeta por cliente (23.191 filas).
- 01_universo_tgt.parquet: clientes del universo con su target.

## Tipos de variables
1. Identity: variables tomadas tal cual de la base.
2. Transform: variables derivadas con lógica de negocio (flags, ratios, antigüedad) y codificación de las categóricas.
3. Aggregate: estadísticos de los 6 meses (suma, máximo, mínimo, promedio, mediana, valores distintos y meses con movimiento).

In [1]:
import pandas as pd
import numpy as np

data_train = pd.read_parquet('../data/processed/02_data_train_limpio.parquet')
data_tarj = pd.read_parquet('../data/processed/02_data_tarj.parquet')
universo = pd.read_parquet('../data/processed/01_universo_tgt.parquet')

print(data_train.shape)   # (139146, 77)
print(data_tarj.shape)    # (23191, 3)
print(universo.shape)     # (23191, 2)

(139146, 77)
(23191, 3)
(23191, 2)


# Identity features
Variables tomadas directamente de la base, sin transformar. Se usa el último mes de entrenamiento (enero 2019) como foto más reciente del cliente.

In [2]:
identity_features = data_train[data_train['Month'] == '2019-01-01'][['client_id', 'SavingAccount_Balance_Average']]
print(identity_features.shape)   # (23191, 2)

(23191, 2)


# Transform features
Se parte de la foto de enero 2019 de cada cliente.

In [3]:
cols_transform = ['client_id', 'Month', 'First_product_dt', 'Last_product_dt',
                  'CreditCard_Premium', 'CreditCard_Active', 'Loan_Active', 'Mortgage_Active',
                  'SavingAccount_Active_ARG_Salary', 'SavingAccount_Active_ARG',
                  'SavingAccount_Active_DOLLAR', 'DebitCard_Active', 'Investment_Active',
                  'Insurance_Life', 'Insurance_Home', 'Insurance_Accidents',
                  'Insurance_Mobile', 'Insurance_ATM', 'Insurance_Unemployment',
                  'Sex', 'Client_Age_grp', 'Mobile', 'Email', 'Region',
                  'SavingAccount_Credits_Amounts', 'SavingAccount_Debits_Amounts']

tf = data_train[data_train['Month'] == '2019-01-01'][cols_transform].copy()
print(tf.shape)   # (23191, 26)

(23191, 26)


## Fechas

In [4]:
for col in ['Month', 'First_product_dt', 'Last_product_dt']:
    tf[col] = pd.to_datetime(tf[col])

## Variables Yes / No a 1 / 0

In [5]:
cols_yes_no = [c for c in tf.columns if set(tf[c].unique()) <= {'Yes', 'No'}]

for col in cols_yes_no:
    tf[col] = tf[col].map({'Yes': 1, 'No': 0})

print(len(cols_yes_no))
print(cols_yes_no)

17
['CreditCard_Premium', 'CreditCard_Active', 'Loan_Active', 'Mortgage_Active', 'SavingAccount_Active_ARG_Salary', 'SavingAccount_Active_ARG', 'SavingAccount_Active_DOLLAR', 'DebitCard_Active', 'Investment_Active', 'Insurance_Life', 'Insurance_Home', 'Insurance_Accidents', 'Insurance_Mobile', 'Insurance_ATM', 'Insurance_Unemployment', 'Mobile', 'Email']


# Variables categóricas
Se trabajan cuatro variables categóricas, cada una con la técnica que mejor se ajusta a su naturaleza:

| Variable | Técnica |
|---|---|
| Region | Porcentaje de target por categoría |
| CreditCard_Product (tipo de tarjeta) | Porcentaje de target por categoría |
| Client_Age_grp (grupo de edad) | Bines ordinales (1 a 7) |
| Sex | Binaria (1 / 0) |

No se usa one-hot encoding: genera una columna por categoría y aumenta la dimensionalidad sin aportar orden ni relación con el target.

El porcentaje de target reemplaza cada categoría por la tasa de clientes con TGT = 1 dentro de esa categoría. Así la variable queda numérica y ordenada según la propensión. Como se calcula con el propio target, en la etapa de modelado debe recalcularse solo con los datos de entrenamiento para evitar data leakage.

## Region

In [6]:
tf['Region'].value_counts()

Region
BUENOS AIRES                     6984
REGION CENTRO                    4579
REGION NORTE GRANDE ARGENTINO    3674
REGION PATAGONICA                2432
CABA Centro/Norte                2048
AMBA Resto                       1828
REGION CUYO                      1640
Sin dato                            6
Name: count, dtype: int64

Los 6 clientes sin región en ningún mes se asignan a la categoría más frecuente (Buenos Aires), ya que no alcanzan para formar una categoría propia con una tasa de target confiable.

In [7]:
moda_region = tf.loc[tf['Region'] != 'Sin dato', 'Region'].mode()[0]
tf['Region'] = tf['Region'].replace('Sin dato', moda_region)
print(moda_region)

BUENOS AIRES


In [8]:
region_tgt = (tf[['client_id', 'Region']]
              .merge(universo, on='client_id')
              .groupby('Region')['TGT']
              .agg(clientes='count', pct_target='mean')
              .sort_values('pct_target', ascending=False))

region_tgt.style.format({'pct_target': '{:.1%}'})

,clientes,pct_target
Region,,
REGION PATAGONICA,2432,33.7%
AMBA Resto,1828,32.4%
BUENOS AIRES,6990,30.6%
CABA Centro/Norte,2048,29.9%
REGION CUYO,1640,29.8%
REGION NORTE GRANDE ARGENTINO,3674,28.7%
REGION CENTRO,4579,24.4%


In [9]:
tf['Region_pct_target'] = tf['Region'].map(region_tgt['pct_target'])
tf[['Region', 'Region_pct_target']].drop_duplicates().sort_values('Region_pct_target', ascending=False)

,Region,Region_pct_target
42,REGION PATAGONICA,0.337171
113,AMBA Resto,0.323851
2,BUENOS AIRES,0.305866
82,CABA Centro/Norte,0.298828
18,REGION CUYO,0.298171
27,REGION NORTE GRANDE ARGENTINO,0.287425
9,REGION CENTRO,0.243721


Conclusión – Region: cada región queda representada por su tasa de clientes que adoptaron el producto, que va de 24,4% (Región Centro) a 33,7% (Región Patagónica). Todas las categorías tienen más de 1.600 clientes, por lo que las tasas son estables.

## Tipo de tarjeta (CreditCard_Product)

In [10]:
data_tarj['CreditCard_Product'].value_counts()

CreditCard_Product
J55660104XX012    8508
No tiene          8168
J55660202XX012    5777
J55660102XX012     402
J55660702XX012     248
J55661002XX012      58
J55660124XX012      29
J55660123XX012       1
Name: count, dtype: int64

Las categorías con menos de 100 clientes se agrupan en Otros, porque con tan pocos casos la tasa de target sería poco confiable.

In [11]:
conteo = data_tarj['CreditCard_Product'].value_counts()
chicas = conteo[conteo < 100].index
data_tarj['CreditCard_Product'] = data_tarj['CreditCard_Product'].replace(list(chicas), 'Otros')

tarj_tgt = (data_tarj.merge(universo, on='client_id')
            .groupby('CreditCard_Product')['TGT']
            .agg(clientes='count', pct_target='mean')
            .sort_values('pct_target', ascending=False))

tarj_tgt.style.format({'pct_target': '{:.1%}'})

,clientes,pct_target
CreditCard_Product,,
Otros,88,53.4%
J55660104XX012,8508,45.7%
J55660702XX012,248,41.9%
J55660202XX012,5777,36.9%
J55660102XX012,402,34.3%
No tiene,8168,6.3%


In [12]:
data_tarj['Tarj_pct_target'] = data_tarj['CreditCard_Product'].map(tarj_tgt['pct_target'])
tarj_features = data_tarj[['client_id', 'Tarj_pct_target']]
print(tarj_features.shape)   # (23191, 2)

(23191, 2)


Conclusión – Tipo de tarjeta: las 8 categorías originales quedan en 6 (las de menos de 100 clientes se agrupan en Otros) y se reemplazan por su tasa de target. Es la categórica con más poder de discriminación: los clientes sin tarjeta de crédito tienen una tasa de 6,3%, mientras que los que tienen tarjeta van de 34,3% a 53,4%.

## Grupo de edad (Client_Age_grp)

In [13]:
tf['Client_Age_grp'].value_counts()

Client_Age_grp
Entre 40 y 49 años    6563
Entre 30 y 39 años    5308
Entre 50 y 59 años    5149
Entre 60 y 64 años    2122
Entre 65 y 69 años    1745
Mayor a 70 años       1315
Entre 18 y 29 años     989
Name: count, dtype: int64

La edad ya viene agrupada en bines. Como los grupos tienen un orden natural, se codifican con números del 1 al 7, conservando ese orden.

In [14]:
orden_edad = {'Entre 18 y 29 años': 1,
              'Entre 30 y 39 años': 2,
              'Entre 40 y 49 años': 3,
              'Entre 50 y 59 años': 4,
              'Entre 60 y 64 años': 5,
              'Entre 65 y 69 años': 6,
              'Mayor a 70 años': 7}

tf['Age_grp_ord'] = tf['Client_Age_grp'].map(orden_edad)

print(tf['Age_grp_ord'].isna().sum())   # tiene que dar 0
tf[['Client_Age_grp', 'Age_grp_ord']].drop_duplicates().sort_values('Age_grp_ord')

0


,Client_Age_grp,Age_grp_ord
53,Entre 18 y 29 años,1
9,Entre 30 y 39 años,2
2,Entre 40 y 49 años,3
32,Entre 50 y 59 años,4
76,Entre 60 y 64 años,5
16,Entre 65 y 69 años,6
132,Mayor a 70 años,7


## Sexo

In [15]:
tf['Sex_F'] = (tf['Sex'] == 'F').astype(int)
tf[['Sex', 'Sex_F']].drop_duplicates()

,Sex,Sex_F
2,F,1
18,M,0


# Creación de nuevas variables (foto de enero)

Cada variable responde una pregunta de negocio sobre el cliente.

## 1. Historia con el banco
- Meses_Antiguedad: hace cuánto es cliente.
- Meses_Ultimo_Producto: hace cuánto no contrata nada.
- Meses_Primer_Ultimo: durante cuánto tiempo fue sumando productos.
- Flag_Contrato_Reciente: 1 si contrató algo en los últimos 6 meses.
- Flag_Producto_Unico: 1 si nunca sumó nada después del primer producto.
- Flag_Cliente_Nuevo: 1 si tiene 12 meses de antigüedad o menos.

In [16]:
tf['Meses_Antiguedad']      = ((tf['Month'] - tf['First_product_dt']).dt.days / 30).clip(lower=0)
tf['Meses_Ultimo_Producto'] = ((tf['Month'] - tf['Last_product_dt']).dt.days / 30).clip(lower=0)
tf['Meses_Primer_Ultimo']   = (tf['Last_product_dt'] - tf['First_product_dt']).dt.days / 30

tf['Flag_Contrato_Reciente'] = (tf['Meses_Ultimo_Producto'] <= 6).astype(int)
tf['Flag_Producto_Unico']    = (tf['First_product_dt'] == tf['Last_product_dt']).astype(int)
tf['Flag_Cliente_Nuevo']     = (tf['Meses_Antiguedad'] <= 12).astype(int)

tf[['Meses_Antiguedad', 'Meses_Ultimo_Producto', 'Meses_Primer_Ultimo',
    'Flag_Contrato_Reciente', 'Flag_Producto_Unico', 'Flag_Cliente_Nuevo']].describe().T

,count,mean,std,min,25%,50%,75%,max
Meses_Antiguedad,23191.0,68.906205,46.905563,0.0,34.083333,57.200000,95.416667,381.500000
Meses_Ultimo_Producto,23191.0,32.162547,44.728615,0.0,1.500000,15.400000,39.433333,381.500000
Meses_Primer_Ultimo,23191.0,36.793788,41.926339,0.0,0.033333,26.966667,55.016667,379.233333
Flag_Contrato_Reciente,23191.0,0.309991,0.462499,0.0,0.000000,0.000000,1.000000,1.000000
Flag_Producto_Unico,23191.0,0.227286,0.419088,0.0,0.000000,0.000000,0.000000,1.000000
Flag_Cliente_Nuevo,23191.0,0.056401,0.230700,0.0,0.000000,0.000000,0.000000,1.000000


Los meses negativos (clientes que contrataron un producto durante enero, después de la fecha de corte del mes) se llevan a 0.

## 2. Qué tiene (vínculo con el banco)

Los seguros figuran todos en No en enero 2019 por falta de carga, así que se toman de diciembre 2018, el último mes con datos.

In [17]:
seguros = ['Insurance_Life', 'Insurance_Home', 'Insurance_Accidents',
           'Insurance_Mobile', 'Insurance_ATM', 'Insurance_Unemployment']

data_train.groupby('Month')[seguros].apply(lambda x: (x == 'Yes').sum())

,Insurance_Life,Insurance_Home,Insurance_Accidents,Insurance_Mobile,Insurance_ATM,Insurance_Unemployment
Month,,,,,,
2018-08-01,0,0,0,0,0,0
2018-09-01,0,0,0,0,0,0
2018-10-01,0,0,0,0,0,0
2018-11-01,0,0,0,0,0,0
2018-12-01,4486,3369,3624,1772,3339,1807
2019-01-01,0,0,0,0,0,0


In [18]:
seguros_dic = data_train[data_train['Month'] == '2018-12-01'][['client_id'] + seguros].copy()
for col in seguros:
    seguros_dic[col] = seguros_dic[col].map({'Yes': 1, 'No': 0})

tf = tf.drop(columns=seguros).merge(seguros_dic, how='left', on='client_id')
print(tf[seguros].isna().sum().sum())   # tiene que dar 0

0


- Cant_Seguros y Flag_Tiene_Seguro: cuántos seguros tiene y si tiene al menos uno.
- Cant_Productos: cuán vinculado está con el banco.
- Cant_Cajas: cuántas cajas de ahorro tiene.
- Flag_Falta_Tarjeta y Flag_Falta_Prestamo: oportunidades comerciales (tiene caja pero no tarjeta; tiene tarjeta pero no préstamo).
- Productos_por_Anio: qué tan rápido suma productos (con un piso de 1 año para no dividir por valores muy chicos).

In [19]:
tf['Cant_Seguros'] = tf[seguros].sum(axis=1)
tf['Flag_Tiene_Seguro'] = (tf['Cant_Seguros'] > 0).astype(int)

productos = ['Loan_Active', 'CreditCard_Active', 'Mortgage_Active',
             'SavingAccount_Active_ARG', 'SavingAccount_Active_DOLLAR',
             'DebitCard_Active', 'Investment_Active']
tf['Cant_Productos'] = tf[productos].sum(axis=1) + tf['Flag_Tiene_Seguro']

cajas = ['SavingAccount_Active_ARG', 'SavingAccount_Active_DOLLAR', 'SavingAccount_Active_ARG_Salary']
tf['Cant_Cajas'] = tf[cajas].sum(axis=1)

tf['Flag_Falta_Tarjeta']  = ((tf['Cant_Cajas'] > 0) & (tf['CreditCard_Active'] == 0)).astype(int)
tf['Flag_Falta_Prestamo'] = ((tf['CreditCard_Active'] == 1) & (tf['Loan_Active'] == 0)).astype(int)

anios = np.maximum(tf['Meses_Antiguedad'] / 12, 1)
tf['Productos_por_Anio'] = tf['Cant_Productos'] / anios

tf[['Cant_Seguros', 'Flag_Tiene_Seguro', 'Cant_Productos', 'Cant_Cajas',
    'Flag_Falta_Tarjeta', 'Flag_Falta_Prestamo', 'Productos_por_Anio']].describe().T

,count,mean,std,min,25%,50%,75%,max
Cant_Seguros,23191.0,0.793282,1.174822,0.0,0.000000,0.000000,1.000000,6.0
Flag_Tiene_Seguro,23191.0,0.409857,0.491818,0.0,0.000000,0.000000,1.000000,1.0
Cant_Productos,23191.0,3.153249,1.556682,0.0,2.000000,3.000000,4.000000,8.0
Cant_Cajas,23191.0,1.322021,0.945089,0.0,0.000000,2.000000,2.000000,3.0
Flag_Falta_Tarjeta,23191.0,0.345177,0.475436,0.0,0.000000,0.000000,1.000000,1.0
Flag_Falta_Prestamo,23191.0,0.558794,0.496542,0.0,0.000000,1.000000,1.000000,1.0
Productos_por_Anio,23191.0,0.950941,0.936669,0.0,0.311217,0.659341,1.219306,7.0


## 3. Cómo maneja la plata
- Prop_Creditos: qué proporción de lo que mueve son ingresos. Reemplaza al ratio créditos / débitos, que explota cuando los débitos son 0.
- Saldo_Neto: cuánto le quedó en el mes (créditos − débitos).
- Flag_Movimientos: 1 si usó la cuenta en el mes.

In [20]:
creditos = tf['SavingAccount_Credits_Amounts']
debitos = tf['SavingAccount_Debits_Amounts']
total = creditos + debitos

tf['Prop_Creditos'] = np.where(total == 0, 0.5, creditos / total)
tf['Saldo_Neto'] = creditos - debitos
tf['Flag_Movimientos'] = ((creditos > 0) | (debitos > 0)).astype(int)

tf[['Prop_Creditos', 'Saldo_Neto', 'Flag_Movimientos']].describe().T

,count,mean,std,min,25%,50%,75%,max
Prop_Creditos,23191.0,0.489033,0.137034,0.00,0.5,0.5,0.5,1.00
Saldo_Neto,23191.0,-389.116601,24833.140127,-1108551.84,0.0,0.0,0.0,1300241.78
Flag_Movimientos,23191.0,0.492476,0.499954,0.00,0.0,0.0,1.0,1.00


## 4. Contacto
- Contactabilidad: por cuántos medios se lo puede contactar (Mobile + Email: 0, 1 o 2).

In [21]:
tf['Contactabilidad'] = tf['Mobile'] + tf['Email']
tf['Contactabilidad'].value_counts()

Contactabilidad
2    15232
1     6795
0     1164
Name: count, dtype: int64

## Se eliminan las columnas auxiliares (fechas y texto original de las categóricas)

In [22]:
auxiliares = ['Month', 'First_product_dt', 'Last_product_dt', 'Sex', 'Client_Age_grp', 'Region']
tf = tf.drop(columns=auxiliares)

print(tf.shape)
print(tf.dtypes.value_counts())

(23191, 40)
int64      31
float64     9
Name: count, dtype: int64


# Aggregate features

## Variables seleccionadas para agregar
Se seleccionaron 25 variables numéricas, siguiendo cuatro criterios:
1. Montos antes que cantidades: cada movimiento (sueldo, transferencias, extracciones, pagos) tiene una variable de monto y otra de cantidad de transacciones, muy correlacionadas entre sí. Se conserva el monto, que además refleja el valor del cliente. Se mantiene solo la cantidad total de transacciones como medida general de actividad.
2. Sin redundancias: se excluye Balance_FirstDate (prácticamente igual al Balance_LastDate del mes anterior) y Total_Amount (suma de créditos y débitos).
3. Sin variables casi vacías: se excluyen las que tienen más del 99% de valores en 0, como pagos en efectivo, IVR y telemarketing.
4. Solo pesos: se excluyen los saldos y consumos en dólares.

In [23]:
# Caja de ahorro (13)
col_sa = ['SavingAccount_Balance_LastDate',
          'SavingAccount_Balance_Average',
          'SavingAccount_Days_with_use',
          'SavingAccount_Transactions_Transactions',
          'SavingAccount_Salary_Payment_Amount',
          'SavingAccount_Transfer_In_Amount',
          'SavingAccount_Transfer_Out_Amount',
          'SavingAccount_ATM_Extraction_Amount',
          'SavingAccount_Service_Payment_Amount',
          'SavingAccount_CreditCard_Payment_Amount',
          'SavingAccount_DebitCard_Spend_Amount',
          'SavingAccount_Credits_Amounts',
          'SavingAccount_Debits_Amounts']

# Tarjeta de crédito (7)
col_cc = ['CreditCard_Balance_ARG',
          'CreditCard_Total_Limit',
          'CreditCard_Total_Spending',
          'CreditCard_Spending_1_Installment',
          'CreditCard_Spending_Installments',
          'CreditCard_Revolving',
          'CreditCard_Payment_Aut_Debit']

# Canales e inversiones (5)
col_ops = ['Operations_Bank',
           'Operations_HomeBanking',
           'Operations_Mobile',
           'Operations_ATM',
           'Investment_Numbers']

print(len(col_sa) + len(col_cc) + len(col_ops))   # 25

25


## Función de agregación
Para cada cliente resume los 6 meses de cada variable con 7 estadísticos: suma, máximo, mínimo, promedio, mediana, cantidad de valores distintos y cantidad de meses con valor distinto de 0.

In [24]:
def agregar(df, columnas, sufijo=''):
    agg = df.groupby('client_id')[columnas].agg(['sum', 'max', 'min', 'mean', 'median', 'nunique'])
    agg.columns = ['_'.join(c) + sufijo for c in agg.columns]

    no_cero = (df[columnas] != 0).groupby(df['client_id']).sum()
    no_cero.columns = [c + '_meses_no_cero' + sufijo for c in no_cero.columns]

    return agg.join(no_cero).reset_index()

## Agregados de los 6 meses

In [25]:
agg_sa = agregar(data_train, col_sa)
agg_cc = agregar(data_train, col_cc)
agg_ops = agregar(data_train, col_ops)

print(agg_sa.shape)    # caja de ahorro
print(agg_cc.shape)    # tarjeta
print(agg_ops.shape)   # canales e inversiones

(23191, 92)
(23191, 50)
(23191, 36)


## Tarjeta en los últimos 3 meses
Capta el comportamiento más reciente con la tarjeta (nov 2018 – ene 2019).

In [26]:
u3m = data_train[data_train['Month'].isin(['2018-11-01', '2018-12-01', '2019-01-01'])]
agg_cc_u3m = agregar(u3m, col_cc, sufijo='_u3m')
print(agg_cc_u3m.shape)

(23191, 50)


## Evolución del revolving
Compara el saldo financiado de la tarjeta entre enero (mes 0), noviembre (mes 2) y agosto (mes 5): la diferencia en pesos y la dirección del cambio (1 subió, −1 bajó, 0 igual).

In [27]:
rev = data_train.pivot(index='client_id', columns='Month', values='CreditCard_Revolving')

m0 = rev['2019-01-01']
m2 = rev['2018-11-01']
m5 = rev['2018-08-01']

comp = pd.DataFrame(index=rev.index)
comp['Rev_0vs2_abs'] = m0 - m2
comp['Rev_0vs5_abs'] = m0 - m5
comp['Rev_2vs5_abs'] = m2 - m5
comp['Rev_0vs2_dir'] = np.sign(m0 - m2)
comp['Rev_0vs5_dir'] = np.sign(m0 - m5)
comp['Rev_2vs5_dir'] = np.sign(m2 - m5)
comp = comp.reset_index()

print(comp.shape)   # (23191, 7)
comp['Rev_0vs5_dir'].value_counts()

(23191, 7)


Rev_0vs5_dir
 0.0    11139
 1.0     6333
-1.0     5719
Name: count, dtype: int64

## Meses con tarjeta y meses con tarjeta premium

In [28]:
cc_meses = data_train[['client_id', 'CreditCard_Active', 'CreditCard_Premium']].copy()
for col in ['CreditCard_Active', 'CreditCard_Premium']:
    cc_meses[col] = (cc_meses[col] == 'Yes').astype(int)

cc_meses = cc_meses.groupby('client_id').sum()
cc_meses.columns = ['Meses_con_tarjeta', 'Meses_premium']
cc_meses = cc_meses.reset_index()

print(cc_meses.shape)   # (23191, 3)
cc_meses['Meses_con_tarjeta'].value_counts()

(23191, 3)


Meses_con_tarjeta
6    15014
0     8116
2       18
1       18
5       17
4        8
Name: count, dtype: int64

# ABT final

In [29]:
ABT = universo[['client_id', 'TGT']] \
    .merge(identity_features, how='left', on='client_id') \
    .merge(tf, how='left', on='client_id') \
    .merge(tarj_features, how='left', on='client_id') \
    .merge(agg_sa, how='left', on='client_id') \
    .merge(agg_cc, how='left', on='client_id') \
    .merge(agg_ops, how='left', on='client_id') \
    .merge(agg_cc_u3m, how='left', on='client_id') \
    .merge(comp, how='left', on='client_id') \
    .merge(cc_meses, how='left', on='client_id')

print('Dimensiones:', ABT.shape)
print('Clientes duplicados:', ABT['client_id'].duplicated().sum())
print('Columnas duplicadas por el merge:', [c for c in ABT.columns if c.endswith(('_x', '_y'))])
print('Valores nulos:', ABT.isnull().sum().sum())
print(ABT.dtypes.value_counts())

Dimensiones: (23191, 275)
Clientes duplicados: 0
Columnas duplicadas por el merge: []
Valores nulos: 0
float64    177
int64       98
Name: count, dtype: int64


In [30]:
resumen_abt = pd.Series({
    'Identity': identity_features.shape[1] - 1,
    'Transform (incluye categóricas)': tf.shape[1] - 1,
    'Tipo de tarjeta': tarj_features.shape[1] - 1,
    'Aggregate caja de ahorro': agg_sa.shape[1] - 1,
    'Aggregate tarjeta': agg_cc.shape[1] - 1,
    'Aggregate canales e inversiones': agg_ops.shape[1] - 1,
    'Aggregate tarjeta últimos 3 meses': agg_cc_u3m.shape[1] - 1,
    'Evolución revolving': comp.shape[1] - 1,
    'Meses con tarjeta': cc_meses.shape[1] - 1,
}, name='variables')
resumen_abt.loc['Total'] = resumen_abt.sum()
resumen_abt.to_frame()

,variables
Identity,1
Transform (incluye categóricas),39
Tipo de tarjeta,1
Aggregate caja de ahorro,91
Aggregate tarjeta,49
Aggregate canales e inversiones,35
Aggregate tarjeta últimos 3 meses,49
Evolución revolving,6
Meses con tarjeta,2
Total,273


In [31]:
ABT.to_parquet('../data/processed/03_abt.parquet', index=False)
ABT.head()

,client_id,TGT,SavingAccount_Balance_Average,CreditCard_Premium,CreditCard_Active,Loan_Active,Mortgage_Active,SavingAccount_Active_ARG_Salary,SavingAccount_Active_ARG,SavingAccount_Active_DOLLAR,...,CreditCard_Revolving_meses_no_cero_u3m,CreditCard_Payment_Aut_Debit_meses_no_cero_u3m,Rev_0vs2_abs,Rev_0vs5_abs,Rev_2vs5_abs,Rev_0vs2_dir,Rev_0vs5_dir,Rev_2vs5_dir,Meses_con_tarjeta,Meses_premium
0,4221845,0,0.00,0,1,0,0,0,0,0,...,0,0,0.00,0.00,0.00,0.0,0.0,0.0,6,0
1,7219504,0,1137.43,0,1,1,0,0,1,0,...,0,0,0.00,0.00,0.00,0.0,0.0,0.0,6,0
2,5882370,0,0.00,1,1,0,0,0,0,0,...,3,0,6394.02,6841.21,447.19,1.0,1.0,1.0,6,6
3,1158013,0,0.00,0,1,0,0,0,0,0,...,3,0,23469.03,1428.11,-22040.92,1.0,1.0,-1.0,6,0
4,3914580,0,545.98,0,0,0,0,0,1,1,...,0,0,0.00,0.00,0.00,0.0,0.0,0.0,0,0


# Conclusión
- La ABT tiene una fila por cliente (23.191 clientes) y todas las variables son numéricas, sin valores nulos.
- Identity: saldo promedio de enero 2019.
- Transform (foto de enero 2019): productos y seguros, antigüedad y fechas en meses, flags de negocio, cantidad de productos y cajas, oportunidades comerciales, proporción de créditos, saldo neto y contactabilidad.
- Categóricas: Region y tipo de tarjeta codificadas con porcentaje de target, grupo de edad con bines ordinales y sexo como binaria.
- Aggregate (6 meses): 25 variables numéricas resumidas con 7 estadísticos, más la tarjeta en los últimos 3 meses, la evolución del revolving y los meses con tarjeta.

Esta es la ABT completa, con la mayor cantidad de variables posibles. La siguiente etapa es la reducción de dimensionalidad (variables casi constantes, correlación e importancia), previa al entrenamiento de los modelos.